Import the three months of flight data and create a database using them with SQLite

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

# 1. List your 3 downloaded CSV file paths
csv_files = [
    "Datasets/Oct25.csv",
    "Datasets/Nov25.csv",
    "Datasets/Dec25.csv"
]

# 2. Create the SQLite database file in your project folder
engine = create_engine("sqlite:///cap4770_flights.db")

# 3. Loop through each CSV, clean BTS formatting quirks, and write to SQLite
for idx, file_path in enumerate(csv_files):
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Could not find file: {file_path}")

    df_month = pd.read_csv(file_path, low_memory=False)

    # Remove the trailing empty 'Unnamed' column caused by BTS trailing commas
    df_month = df_month.loc[:, ~df_month.columns.str.contains("^Unnamed")]
    df_month.columns = [col.strip().upper() for col in df_month.columns]

    # Overwrite table on the 1st file ('replace'), then add the 2nd and 3rd ('append')
    mode = "replace" if idx == 0 else "append"
    df_month.to_sql(
        name="bts_flights_raw",
        con=engine,
        if_exists=mode,
        index=False,
        chunksize=10000
    )
    print(f"Loaded {len(df_month):,} rows from {os.path.basename(file_path)} (mode='{mode}')")

print("\nAll 3 files successfully saved to cap4770_flights.db!")

Loaded 668,332 rows from Oct25.csv (mode='replace')
Loaded 630,188 rows from Nov25.csv (mode='append')
Loaded 644,987 rows from Dec25.csv (mode='append')

All 3 files successfully saved to cap4770_flights.db!


In [2]:
verify_query = text("""
    SELECT MONTH, COUNT(*) AS total_flights
    FROM bts_flights_raw
    GROUP BY MONTH
    ORDER BY MONTH;
""")

with engine.connect() as conn:
    summary = pd.read_sql(verify_query, con=conn)

print(summary)

   MONTH  total_flights
0     10         668332
1     11         630188
2     12         644987
